<a href="https://colab.research.google.com/github/Harshavardhan655/complier-design-/blob/main/cd_wek_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!apt-get update -qq
!apt-get install -y flex gcc


In [ ]:

%%writefile vehicle.l
%{
#include <stdio.h>
%}

%%
[A-Z]{2}[0-9]{2}[A-Z]{1,2}[0-9]{4} {
    printf("VALID VEHICLE: %s\n", yytext);
}

[A-Z][A-Z0-9]* {
    printf("INVALID: %s\n", yytext);
}

[ \t\r\n]+  ;
[0-9]+ {
    printf("INVALID: %s\n", yytext);
}
. ;
%%

int main(void)
{
    printf("Enter vehicle numbers:\n");
    yylex();
    return 0;
}

int yywrap(void) { return 1; }


In [ ]:

!flex vehicle.l
!gcc lex.yy.c -o vehicle -lfl
!echo 'TS09EA1234 KA5AB123 AP31AB9876 MH12A1234' | ./vehicle


In [ ]:

%%writefile analyzer.l
%{
#include <stdio.h>
#include <string.h>

int keywords = 0, identifiers = 0, numbers = 0;
int operators = 0, symbols = 0, lines = 0;

int is_keyword(char *s)
{
    char *kw[] = {
        "int", "float", "char", "if", "else",
        "while", "for", "return", "void", "double",
        "break", "continue", "switch", "case"
    };

    for (int i = 0; i < 14; i++)
        if (strcmp(s, kw[i]) == 0)
            return 1;

    return 0;
}
%}

%%
"int"|"float"|"char"|"if"|"else"|"while"|"for"|"return"|"void"|"double"|"break"|"continue"|"switch"|"case"
    { keywords++; }

[a-zA-Z_][a-zA-Z0-9_]* {
    if (is_keyword(yytext))
        keywords++;
    else
        identifiers++;
}

[0-9]+(\.[0-9]+)?  { numbers++; }

"=="|"!="|"<="|">="|"++"|"--"|"&&"|"||"|"+="|"-="|"*="|"/="|"="|"+"|"-"|"*"|"/"|"%"|"<"|">"
    { operators++; }

[(){}\[\],;.] { symbols++; }

[ \t\r]+  ;
\n         { lines++; }
.          ;
%%

int main(void)
{
    printf("Enter C source code (Ctrl+D to finish):\n");
    yylex();

    printf("\n--- Lexical Statistics ---\n");
    printf("Keywords        : %d\n", keywords);
    printf("Identifiers     : %d\n", identifiers);
    printf("Numbers         : %d\n", numbers);
    printf("Operators       : %d\n", operators);
    printf("Special Symbols : %d\n", symbols);
    printf("Lines           : %d\n", lines);

    return 0;
}

int yywrap(void) { return 1; }
